# EDA CHR&R Dataset 

## Introduction
This document provides an exploratory data analysis (EDA) of the CHR&R dataset. The goal of this analysis is to understand the structure, characteristics, and potential insights that can be derived from the dataset. The analysis will include data cleaning, visualization, and statistical summaries to identify patterns and relationships within the data.


# Import Libraries 

In [ ]:
import pandas as pd
from pathlib import Path 

# Read The CSV file 

In [2]:
base_path = Path.cwd().parents[1]  # Get the parent directory of the current working directory

data_path = base_path / "data" / "raw"/ "analytic_data2025.csv"

df = pd.read_csv(data_path)

print(f"Data loaded successfully")

Data loaded successfully


<positron-console-cell-2>:5: DtypeWarning: Columns (0,1,2,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,26

The warning is saying that Pandas is unsure if those columns are numbers or text. 

In [3]:
# Force Pandas to read the entire file (or a larger chunk of it) before deciding on the data types
df_chr = pd.read_csv(data_path, low_memory=False)

# Exploratory Data Analysis (EDA)

In [4]:
df_chr.head()

,State FIPS Code,County FIPS Code,5-digit FIPS Code,State Abbreviation,Name,Release Year,County Clustered (Yes=1/No=0),Premature Death raw value,Premature Death numerator,Premature Death denominator,...,% Rural raw value,% Rural numerator,% Rural denominator,% Rural CI low,% Rural CI high,Population raw value,Population numerator,Population denominator,Population CI low,Population CI high
0,statecode,countycode,fipscode,state,county,year,county_clustered,v001_rawvalue,v001_numerator,v001_denominator,...,v058_rawvalue,v058_numerator,v058_denominator,v058_cilow,v058_cihigh,v051_rawvalue,v051_numerator,v051_denominator,v051_cilow,v051_cihigh
1,00,000,00000,US,United States,2025,NaN,8351.7365494,4763989,925367214,...,0.2000313707,66300254,331449281,NaN,NaN,334914895,NaN,NaN,NaN,NaN
2,01,000,01000,AL,Alabama,2025,NaN,11853.247248,102760,13958454,...,0.4226276049,2123399,5024279,NaN,NaN,5108468,NaN,NaN,NaN,NaN
3,01,001,01001,AL,Autauga County,2025,1,9938.2633823,1008,163064,...,0.406768132,23920,58805,NaN,NaN,60342,NaN,NaN,NaN,NaN
4,01,003,01003,AL,Baldwin County,2025,1,8957.1126859,3944,653515,...,0.3758645536,87113,231767,NaN,NaN,253507,NaN,NaN,NaN,NaN


The FIPS columns have statecode, countycode, and fipscode.

    fipscode is the one to keep for "Join Key."

    Warning: 1001 is missing the leading zero for Alabama (Autauga County). Merging this with a dataset that has 01001, the join will fail.

Wide Format: This dataset is extremely wide (hundreds of columns like v001_rawvalue, v058_rawvalue, etc.). This is common for CHR&R, where they use a "v-code" system for different health measures.

Data Types: Have a mix of strings ("Alabama") and numeric data.

Row Zero seems to duplucate the names with "v-code" headers. Keep row 0 as the header and skip the current human-readable header row.

In [5]:
# Print the first few rows to see the repetition
print(df_chr.head(3))

  State FIPS Code County FIPS Code 5-digit FIPS Code State Abbreviation  \
0       statecode       countycode          fipscode              state   
1              00              000             00000                 US   
2              01              000             01000                 AL   

            Name Release Year County Clustered (Yes=1/No=0)  \
0         county         year              county_clustered   
1  United States         2025                           NaN   
2        Alabama         2025                           NaN   

  Premature Death raw value Premature Death numerator  \
0             v001_rawvalue            v001_numerator   
1              8351.7365494                   4763989   
2              11853.247248                    102760   

  Premature Death denominator  ... % Rural raw value % Rural numerator  \
0            v001_denominator  ...     v058_rawvalue    v058_numerator   
1                   925367214  ...      0.2000313707          6630025

In [6]:
# Force fipscode to be a 5-digit string, keeping leading zeros
# 'header=1' tells pandas to Skip row 0 (human names) and use row 1 (v-codes) as the column names.
df = pd.read_csv(
    data_path,
    header=1, 
    dtype={'fipscode': str}, 
    low_memory=False
)

# Pad with a leading zero if it's only 4 digits (e.g., '1001' -> '01001')
df['fipscode'] = df['fipscode'].str.zfill(5)

# Verify the fix
print(df['fipscode'].head())

0    00000
1    01000
2    01001
3    01003
4    01005
Name: fipscode, dtype: object


In [7]:
df.head()

,statecode,countycode,fipscode,state,county,year,county_clustered,v001_rawvalue,v001_numerator,v001_denominator,...,v058_rawvalue,v058_numerator,v058_denominator,v058_cilow,v058_cihigh,v051_rawvalue,v051_numerator,v051_denominator,v051_cilow,v051_cihigh
0,0,0,00000,US,United States,2025,NaN,8351.736549,4763989.0,925367214.0,...,0.200031,66300254.0,331449281.0,NaN,NaN,334914895.0,NaN,NaN,NaN,NaN
1,1,0,01000,AL,Alabama,2025,NaN,11853.247248,102760.0,13958454.0,...,0.422628,2123399.0,5024279.0,NaN,NaN,5108468.0,NaN,NaN,NaN,NaN
2,1,1,01001,AL,Autauga County,2025,1.0,9938.263382,1008.0,163064.0,...,0.406768,23920.0,58805.0,NaN,NaN,60342.0,NaN,NaN,NaN,NaN
3,1,3,01003,AL,Baldwin County,2025,1.0,8957.112686,3944.0,653515.0,...,0.375865,87113.0,231767.0,NaN,NaN,253507.0,NaN,NaN,NaN,NaN
4,1,5,01005,AL,Barbour County,2025,1.0,12738.656137,587.0,67912.0,...,0.659200,16627.0,25223.0,NaN,NaN,24585.0,NaN,NaN,NaN,NaN


In [8]:
df.shape

(3204, 796)

In [9]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
statecode,3204.0,30.181024,1.518720e+01,0.000000,18.000000,29.000000,45.000000,5.600000e+01
countycode,3204.0,101.948814,1.074630e+02,0.000000,33.000000,77.000000,133.000000,8.400000e+02
year,3204.0,2025.000000,0.000000e+00,2025.000000,2025.000000,2025.000000,2025.000000,2.025000e+03
county_clustered,3152.0,0.980013,1.399787e-01,0.000000,1.000000,1.000000,1.000000,1.000000e+00
v001_rawvalue,3140.0,10367.292644,3.813656e+03,3315.252949,7719.617819,9798.223368,12344.826037,4.641785e+04
...,...,...,...,...,...,...,...,...
v051_rawvalue,3196.0,314375.683667,6.057055e+06,43.000000,10927.000000,26713.000000,75649.500000,3.349149e+08
v051_numerator,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
v051_denominator,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
v051_cilow,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Uniformity:  3,204 rows, which is consistent with the number of counties in the U.S. (plus the state/national aggregates).

The Red Flags (Data Health)

    Empty/Null Columns: Look at v051_numerator, v051_denominator, etc. The count is 0.0 and the rest are NaN.

        What this means: These columns contain no data. They are essentially "dead weight."

        Action: You should drop these columns during your cleaning phase to keep your dataframe lightweight.

    Scientific Notation: You see numbers like 5.600000e+01. This is Pandas shorthand for scientific notation. This is fine for now, but keep an eye on it if you try to perform precise matching, as it can sometimes hide leading zeros if not handled as a string.

In [10]:
df.isnull().sum()

statecode              0
countycode             0
fipscode               0
state                  0
county                 0
                    ... 
v051_rawvalue          8
v051_numerator      3204
v051_denominator    3204
v051_cilow          3204
v051_cihigh         3204
Length: 796, dtype: int64

In [11]:
null_counts = df_chr.isnull().sum()
# Show only columns where null count > 3000
print(null_counts[null_counts > 3000])

Premature Death (NHOPI)                                                             3109
Premature Death CI low (NHOPI)                                                      3109
Premature Death CI high (NHOPI)                                                     3109
Premature Death flag (Two or more races) (. = No Flag/1=Unreliable/2=Suppressed)    3204
Poor Physical Health Days numerator                                                 3204
                                                                                    ... 
% Rural CI high                                                                     3204
Population numerator                                                                3204
Population denominator                                                              3204
Population CI low                                                                   3204
Population CI high                                                                  3204
Length: 242, dtype: i

Column counts like 3109 or 3204 nulls (out of 3204 rows), are likely "sparse" data. In the context of the County Health Rankings, these columns usually fall into two categories:

    Sub-population specific data: Fields like "NHOPI" (Native Hawaiian or Other Pacific Islander) are often suppressed or unavailable for counties with very small populations to protect privacy.

    Metadata/Formatting errors: Columns like "Population numerator" being completely empty often mean that specific raw data point wasn't captured in the standardized format for this release.

Drop them since they will provide no statistical power and introduce noise into your analysis.

In [12]:
# Keep only columns where less than 3000 are missing
# This removes the "empty" columns while keeping the valid ones
df_clean = df_chr.loc[:, df_chr.isnull().sum() < 3000].copy()

# Print the shape to see how much "weight" you've shed
print(f"Original shape: {df_chr.shape}")
print(f"Cleaned shape: {df_clean.shape}")

Original shape: (3205, 796)
Cleaned shape: (3205, 554)


In [13]:
df_clean.isnull().sum()

State FIPS Code                                  0
County FIPS Code                                 0
5-digit FIPS Code                                0
State Abbreviation                               0
Name                                             0
                                                ..
Children in Single-Parent Households CI high    10
% Rural raw value                                8
% Rural numerator                                8
% Rural denominator                              8
Population raw value                             8
Length: 554, dtype: int64

In [15]:
df_clean.dtypes

State FIPS Code                                 object
County FIPS Code                                object
5-digit FIPS Code                               object
State Abbreviation                              object
Name                                            object
                                                 ...  
Children in Single-Parent Households CI high    object
% Rural raw value                               object
% Rural numerator                               object
% Rural denominator                             object
Population raw value                            object
Length: 554, dtype: object

In [19]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3205 entries, 0 to 3204
Columns: 554 entries, State FIPS Code to Population raw value
dtypes: object(554)
memory usage: 13.5+ MB


This dataset has wide data with thousands of potential variables, but many of them are either empty or not useful for analysis. Focus on the columns that have meaningful data and can provide insights into health outcomes and determinants.